## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-16-19-50-12 +0000,wapo,National service promoted as a bipartisan solu...,https://www.washingtonpost.com/nation/2026/11/...
1,2026-11-16-19-43-53 +0000,nyt,Winter Nightmare Looms Early in Ukraine as Rus...,https://www.nytimes.com/2026/11/16/world/europ...
2,2026-11-16-19-39-34 +0000,cbc,Here are the 3 major immigration cases U.S. Su...,https://www.cbc.ca/news/world/us-scotus-major-...
3,2026-11-16-19-33-21 +0000,nypost,OpenAI sued by California after 80 incidents t...,https://nypost.com/2026/11/16/us-news/californ...
4,2026-11-16-19-33-00 +0000,wsj,Kalshi Buys Securities Firm to Expand Into Sto...,https://www.wsj.com/finance/kalshi-buys-securi...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2750/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
115,trump,31
147,nvidia,22
54,new,21
100,china,16
253,ai,15


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
70,2026-11-16-17-24-00 +0000,wsj,President Trump said he might meet with North ...,https://www.wsj.com/politics/trump-says-he-mig...,107
151,2026-11-16-14-46-32 +0000,nypost,Trump ‘would not mind’ Jensen Huang reinstated...,https://nypost.com/2026/11/16/us-news/trump-wo...,91
58,2026-11-16-17-48-36 +0000,nyt,Trump Says He’s Open to Reinstatement of Jense...,https://www.nytimes.com/2026/11/16/us/politics...,88
62,2026-11-16-17-43-00 +0000,wsj,Nvidia Scandal Threatens China’s Charm Offensi...,https://www.wsj.com/world/china/nvidia-scandal...,87
120,2026-11-16-15-50-59 +0000,nypost,Trump defends Commerce boss Lutnick over Nvidi...,https://nypost.com/2026/11/16/us-news/trump-de...,86


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
70,107,2026-11-16-17-24-00 +0000,wsj,President Trump said he might meet with North ...,https://www.wsj.com/politics/trump-says-he-mig...
324,76,2026-11-15-20-20-57 +0000,nypost,Jensen Huang eyeing comeback as Nvidia CEO for...,https://nypost.com/2026/11/15/business/jensen-...
188,61,2026-11-16-12-03-41 +0000,chitri,Questions swirl around FAA’s new SMART AI syst...,https://www.chicagotribune.com/2026/11/16/faa-...
187,46,2026-11-16-12-04-00 +0000,wsj,A Texas-based U.S. attorney sent what he calle...,https://www.wsj.com/arts-culture/books/the-whi...
5,42,2026-11-16-19-30-30 +0000,bbc,Lindsay Clancy to face retrial next year for d...,https://www.bbc.co.uk/news/articles/cw5y1v8de3...
175,37,2026-11-16-13-10-23 +0000,cbc,Ukraine launches biggest drone attack on Russi...,https://www.cbc.ca/news/world/ukraine-russia-d...
36,34,2026-11-16-18-32-20 +0000,nypost,"Chase, Bank of America, Wells Fargo and Capita...",https://nypost.com/2026/11/16/business/outages...
299,32,2026-11-15-22-33-03 +0000,nypost,Major Red Sea strait falls under full Houthi c...,https://nypost.com/2026/11/15/world-news/major...
103,32,2026-11-16-16-18-54 +0000,nypost,Terrifying moment 30ft El Niño-powered wave wi...,https://nypost.com/2026/11/16/us-news/massive-...
209,31,2026-11-16-11-00-00 +0000,nypost,NYC uses twice as many delivery contractors as...,https://nypost.com/2026/11/16/business/nyc-use...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
